In [28]:

import os 
import sys
import pandas as pd
import json
import re
import time
from psycopg2 import sql, Error
from datetime import datetime, timezone

from extractors import *
from utils.utils import *

from dotenv import load_dotenv
import os
import sys



load_dotenv()

conn_params = json.loads(os.getenv('CONN_PARAMS', '{}'))


Se extraen los datos de la primera función products_info_for_collection() que devuelve un dataframe con los links y productos por cada colección 

In [2]:
df_products_info_for_collection = products_info_for_collection()

In [8]:
df_products_info_for_collection.sample(5)

,collection_name,product_name,product_link,product_image,price
27,face,PhotoReady Rose Glow™ Face Mist,https://www.revlon.com/products/photoready-ros...,https://www.revlon.com/cdn/shop/files/30997012...,$16.79
22,face,ColorStay™ Lock Setting Mist,https://www.revlon.com/products/colorstay-lock...,https://www.revlon.com/cdn/shop/files/30997016...,$14.99
25,face,PhotoReady Prime Plus™ Makeup and Skincare Pri...,https://www.revlon.com/products/photoready-pri...,https://www.revlon.com/cdn/shop/files/revlon-f...,$16.99
7,face,ColorStay™ Longwear Makeup SPF 20,https://www.revlon.com/products/colorstay-long...,https://www.revlon.com/cdn/shop/files/RV_2H22_...,$16.99
18,face,Illuminance™ Gel Serum Blush,https://www.revlon.com/products/illuminance-ge...,https://www.revlon.com/cdn/shop/files/revlon-e...,$14.99


In [4]:


df_products_info_for_collection['product_name'] = df_products_info_for_collection.apply(
    lambda row: re.search(r"[^/]+$", row['product_link']).group() if row['product_name'] == '' else row['product_name'],
    axis=1
)

In [ ]:
details_products(df_products_info_for_collection[:3], conn_params )

La Tabla de los Registros Insertados se ve de la siguiente forma

In [8]:
def get_data_sql(conn_params, query):
    """Obtiene los registros de Postgres y devuelve un df"""
    
    try:
        with get_db_connection(conn_params) as conn:
            with conn.cursor() as cursor:  
                return pd.read_sql(query, conn)   
    except Error as e:
        print(f"Error al consultar: {e}")
    
    return 


In [ ]:
query = "SELECT * FROM products_new ORDER BY RANDOM() LIMIT 5"
df_details_products =  get_data_sql(conn_params, query)

C:\Users\Dell\AppData\Local\Temp\ipykernel_1132\3545591060.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql("SELECT * FROM products_new ORDER BY RANDOM() LIMIT 5", conn)


In [37]:
pd.set_option('display.max_colwidth', None)
df_details_products

,id,product_name,product_link,product_link_img,price,description,counts_reviews,star_1,star_2,star_3,star_4,star_5,collection_id
0,26,ColorStay™ Liquid Eyeliner,https://www.revlon.com/products/colorstay-liquid-eyeliner,https://www.revlon.com/cdn/shop/files/309974209021.jpg?v=1731727279&width=1080,$10.99,Our classic ColorStay™ Liquid Liner allows you to emphasize your eyes with ultra-fine or dramatically bold looks that last\nup to 16 hours.,2397,46,95,276,634,1346,1
1,19,ColorStay Full Cover™ Foundation,https://www.revlon.com/products/colorstay-full-cover-foundation,https://www.revlon.com/cdn/shop/files/309971335013_1.jpg?v=1731727337&width=1080,$16.99,An effortless longwear fluid foundation with buildable coverage that looks flawless and feels fresh for 12 hours. The lightweight liquid perfects just the right amount. It evens out complexions with a range of adaptable shades that disappear into most skin tones and hydrates skin all day.,1455,69,52,166,386,782,2
2,84,Super Lustrous™ Glimmer Gloss,https://www.revlon.com/products/super-lustrous-glimmer-gloss,https://www.revlon.com/cdn/shop/files/RV_1H25_SL_Glimmer_Gloss_SoldierApplicator_Happy_Hour_004_300RGB.3000x3000.jpg?v=1761157811&width=1080,$11.99,"Glimmer. Shimmer. Shine. Revlon Super Lustrous Glimmer Gloss is the first lip gloss made with plant-based glitter and light-refracting pearls for shimmering, multi-dimensional shine with every swipe. Non-gritty, non-sticky feel. The luxe, lightweight formula glides on smoothly, gives lips a fuller and plumper appearance, plus instantly boosts hydration by up to 39%. Formulated with a Botanical Complex of Agave, Moringa + Cupuacu Butter. Available in 10 glimmering shades.",1329,12,19,125,333,840,4
3,13,ColorStay™ Brow Pencil,https://www.revlon.com/products/colorstay-brow-pencil-with-blending-eyebrow-brush,https://www.revlon.com/cdn/shop/files/revlon-ecomm-ATF-eye-CS_Brow_Pencil_SoldierBulk_Blonde_205-3000x3000.3000x3000.jpg?v=1762457555&width=1080,$12.99,"Define, shape, and enhance your brows with the Revlon ColorStay Eyebrow Pencil with an angled precision tip and built-in spoolie brush. The slanted tip on this long-wearing, waterproof eyebrow pencil glides on smoothly with hair-like strokes for precise application and buildable color that lasts up to 24 hours. Use the soft brow brush to blend and diffuse color, giving eyebrows a polished look. Define your brows and fill in sparse areas with soft, natural eyebrow color to serve bold looks that will last all day.",1196,33,36,108,313,706,1
4,32,ColorStay Matte Lite Crayon™ Lipstick,https://www.revlon.com/products/colorstay-matte-lite-crayon-lipstick,https://www.revlon.com/cdn/shop/files/revlon-ecomm-ATF-lip-colorstay-matte-lite-crayon-soldier-air-kiss-010-3000x3000.3000x3000.jpg?v=1733500354&width=1080,$12.39,"30% lighter than your average lipstick. ColorStay Matte Lite Crayon™ is so lightweight you’ll forget it’s even on. Experience saturated color in a comfortable, non-drying matte formula infused with antioxidant-rich Mango Seed Oil to help replenish lips. In 12 bold yet wearable shades for color play all day.",1769,26,41,178,457,1067,4


In [9]:
query = "SELECT * FROM product_store_prices_new ORDER BY RANDOM() LIMIT 5"
df_prices_stores =  get_data_sql(conn_params, query)

C:\Users\Dell\AppData\Local\Temp\ipykernel_12580\4148385232.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


In [10]:
df_prices_stores

,id,product_id,store_name,store_price,updated_at
0,168,87,Amazon.com,$6.48,2026-04-08 16:32:54.787782
1,44,42,Walmart,$10.48,2026-04-08 16:32:54.787782
2,195,59,Target,$5.99,2026-04-08 16:32:54.787782
3,258,15,CVS,$11.69,2026-04-08 16:32:54.787782
4,205,37,Target,$10.29,2026-04-08 16:32:54.787782


Finalmente se extraen los comentarios

In [32]:
comments_products(df_products_info_for_collection,conn_params)

Error al extraer comentarios - https://www.revlon.com/products/photoready-lift-fill-skin-tint
Error al extraer comentarios - https://www.revlon.com/products/photoready-lift-fill-skin-tint
El botón está deshabilitado. ¡Llegamos al final!
(301, 7) https://www.revlon.com/products/photoready-lift-fill-skin-tint
Registros insertados correctamente. 298


,product_id,title_comments,author_comments,rating_comments,message_comments,dateCreated,page
0,64,Color isnt what it was or what they say it is,VivianN,1,I have bought the naughty nudes for over 10 yr...,2026-04-08,1
1,64,"Ultra fine , smooth , easy to blend ,the natur...",YugviP,5,This issue a pressed powder blush with super f...,2026-02-24,1
2,64,"Pretty, Blendable Blush",MagdalinaI,3,"they have cute shades, easy to blend and is we...",2026-02-24,1
3,64,Natural and smooth,DawnJadeC,5,Tried recently and I’m very happy. texture is ...,2026-02-23,1
4,64,throw it in the bag and go,,5,I love the color pay off and if is so convenie...,2026-02-22,1
...,...,...,...,...,...,...,...
296,64,What a waste,,1,First time purchase and sadly will be the last...,2019-06-18,60
297,64,Waste of money,,1,This product has no pigment what so ever on th...,2019-05-31,60
298,64,easy to blend,,4,I have purchased this product for a couple of ...,2019-03-15,61
299,64,Don’t purchase,,1,Barely shows up. Crumbled soon after I purchas...,2019-03-13,61
